In [1]:
import pandas as pd

human_path = "election2024_tweets.csv"
ai_path    = "synthetic_election2024.csv"

human_df = pd.read_csv(human_path)
ai_df    = pd.read_csv(ai_path)

print("Human columns:", human_df.columns)
print("AI columns:", ai_df.columns)

human_df.head()

Human columns: Index(['Tweet ID', 'Tweet_count', 'Username', 'Text', 'Created At', 'Retweets',
       'Likes', 'User Bio', 'Follower Count', 'Following Count', 'Replies',
       'Location', 'Age', 'Gender', 'Hashtags'],
      dtype='object')
AI columns: Index(['Text'], dtype='object')


,Tweet ID,Tweet_count,Username,Text,Created At,Retweets,Likes,User Bio,Follower Count,Following Count,Replies,Location,Age,Gender,Hashtags
0,1857211912523850191,1,Andrew Olson (PCO) ⚔️⚔️⚔️,"""Trump stole it"", says dollar-store Obama.\n\n...",Thu Nov 14 23:59:59 +0000 2024,0,1,🇺🇸🍊⚔\nConservative. Former Dem who broke the ...,1350,1084,0,United States,Unknown,Unknown,[]
1,1857211911802114113,2,Sid,@SpeakerJohnson @SenJohnThune I think you need...,Thu Nov 14 23:59:59 +0000 2024,0,0,NaN,37,88,0,Unknown,Unknown,Unknown,[]
2,1857211911713988946,3,🇺🇸 Alf 🇺🇸,@EricLDaugh Insane the 10 GOP Senators all vot...,Thu Nov 14 23:59:59 +0000 2024,0,1,RIP Charlie Kirk 🙏🏻🇺🇸,2386,1664,0,"MAGA, USA 🇺🇸",Unknown,Unknown,[]
3,1857211911626187170,4,Kathy,@TristanSnell 1- This is like his 50th federal...,Thu Nov 14 23:59:59 +0000 2024,15,51,"I love Big Brother, Hockey and to give my 2 cents",13401,12274,20,"Montreal, QC, Canada",Unknown,Unknown,[]
4,1857211911567233305,5,National Catholic Register,What the @USCCB Meeting Said About the US Bish...,Thu Nov 14 23:59:59 +0000 2024,0,4,"Our mission is to inform, inspire, challenge a...",209892,727,0,United States,Unknown,Unknown,[]


In [2]:
import random
import re
import pandas as pd

TEXT_COL = "Text"  



def random_typo(word, p=0.15):
    if len(word) < 4 or random.random() > p:
        return word
    i = random.randint(0, len(word)-2)
    # swap two adjacent chars
    return word[:i] + word[i+1] + word[i] + word[i+2:]

def add_slang(text):
    slang_map = {
        "you": "u",
        "people": "ppl",
        "really": "rlly",
        "because": "cuz",
        "are": "r",
        "before": "b4",
    }
    for k, v in slang_map.items():
        # replace whole words only
        text = re.sub(rf"\b{k}\b", v, text, flags=re.IGNORECASE)
    return text

def messy_punctuation(text):
    # randomly remove or add !!! and ???
    if random.random() < 0.5:
        text = re.sub(r"[.!?]+", "", text)      # remove punctuation
    else:
        text = text + random.choice(["!!!", "??", "?!?"])
    return text

def elongate_words(text):
    def _elongate(match):
        ch = match.group(0)
        return ch * random.randint(3, 6)
    # elongate some vowels
    return re.sub(r"[aeiouAEIOU]", _elongate, text, count=1)

def simple_ideology_flip(text):
    flip_map = {
        "support": "oppose",
        "oppose": "support",
        "good": "bad",
        "bad": "good",
        "right": "wrong",
        "left": "right",
    }
    for k, v in flip_map.items():
        text = re.sub(rf"\b{k}\b", v, text, flags=re.IGNORECASE)
    return text

def adversarial_perturb(text):
    # pick a random subset of attacks
    funcs = [random_typo_words, add_slang, messy_punctuation, elongate_words, simple_ideology_flip]
    random.shuffle(funcs)
    for f in funcs[: random.randint(1, 3)]:
        text = f(text)
    return text

def random_typo_words(text):
    words = text.split()
    words = [random_typo(w) for w in words]
    return " ".join(words)


In [3]:
def make_adversarial_df(ai_df, n_aug_per_row=1):
    rows = []
    for _, row in ai_df.iterrows():
        base_text = row[TEXT_COL]
        for _ in range(n_aug_per_row):
            adv_text = adversarial_perturb(base_text)
            rows.append({TEXT_COL: adv_text})
    adv_df = pd.DataFrame(rows)
    return adv_df

ai_adv_df = make_adversarial_df(ai_df, n_aug_per_row=1)

ai_df_clean = ai_df[[TEXT_COL]].copy()
ai_df_clean["label"] = 1
ai_df_clean["source"] = "ai_clean"

ai_adv_df["label"] = 1
ai_adv_df["source"] = "ai_adv"


In [4]:
human_df_sub = human_df[[TEXT_COL]].copy()
human_df_sub["label"] = 0
human_df_sub["source"] = "human_scraped"


In [5]:
df_all = pd.concat([human_df_sub, ai_df_clean, ai_adv_df], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)
df_all["source"].value_counts()


source
ai_adv           100935
ai_clean         100935
human_scraped    100000
Name: count, dtype: int64

In [6]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_all, test_size=0.2, random_state=42, stratify=df_all["label"]
)

train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))
len(train_ds), len(test_ds)

/home/gg2713/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(241496, 60374)

In [7]:
from transformers import AutoTokenizer

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=128,
    )

train_tok = train_ds.map(tokenize_batch, batched=True)
test_tok  = test_ds.map(tokenize_batch, batched=True)

# rename 'label' -> 'labels' and set format
train_tok = train_tok.rename_column("label", "labels")
test_tok  = test_tok.rename_column("label", "labels")

train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])



Map: 100%|██████████| 60374/60374 [00:05<00:00, 11768.65 examples/s]


In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to("cuda")


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

In [9]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./roberta_elec2024",
    eval_strategy="epoch",         # old HuggingFace API
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics,
)

trainer.train()


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.020600,0.003015,0.999636,0.999505,0.999950,0.999728
2,0.000000,0.001496,0.999834,0.999777,0.999975,0.999876
3,0.000000,0.000013,0.999983,0.999975,1.000000,0.999988


TrainOutput(global_step=90561, training_loss=0.004982771282495429, metrics={'train_runtime': 6672.7648, 'train_samples_per_second': 108.574, 'train_steps_per_second': 13.572, 'total_flos': 4.765520056891392e+16, 'train_loss': 0.004982771282495429, 'epoch': 3.0})

In [13]:
metrics = trainer.evaluate()
metrics

{'eval_loss': 1.2787866580765694e-05,
 'eval_accuracy': 0.9999834365786597,
 'eval_precision': 0.9999752321981424,
 'eval_recall': 1.0,
 'eval_f1': 0.9999876159457083,
 'eval_runtime': 151.7928,
 'eval_samples_per_second': 397.739,
 'eval_steps_per_second': 49.719,
 'epoch': 3.0}

In [14]:
import numpy as np
from sklearn.metrics import classification_report

# Predict on test set
preds = trainer.predict(test_tok)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)

# test_df still corresponds to test_tok rows
sources = test_df["source"].values

print("Overall:")
print(classification_report(y_true, y_pred))

for src in ["human_scraped", "ai_clean", "ai_adv"]:
    mask = (sources == src)
    print(f"\nSource = {src}")
    print(classification_report(y_true[mask], y_pred[mask]))


Overall:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20000
           1       1.00      1.00      1.00     40374

    accuracy                           1.00     60374
   macro avg       1.00      1.00      1.00     60374
weighted avg       1.00      1.00      1.00     60374


Source = human_scraped
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20000
           1       0.00      0.00      0.00         0

    accuracy                           1.00     20000
   macro avg       0.50      0.50      0.50     20000
weighted avg       1.00      1.00      1.00     20000


Source = ai_clean
              precision    recall  f1-score   support

           1       1.00      1.00      1.00     20123

    accuracy                           1.00     20123
   macro avg       1.00      1.00      1.00     20123
weighted avg       1.00      1.00      1.00     20123


Source = ai_adv
    

/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/gg2713/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
